# QM 640 Capstone — Step 3c: Automated Objective Flags

This notebook fills in the columns of `screening_worksheet.csv` that are
**objective data lookups, not judgment calls**: `confounding_event_flag`
(does this filer have another filing within +/-2 days?) and
`sufficient_history_flag` (does the firm have >=120 trading days of price
history before the event?). It also flags suspicious price-data gaps for
you to eyeball as a possible trading halt, and auto-populates
`exclude_reason` only where an objective flag already justifies it.

**What this notebook deliberately does NOT touch:** `is_genuine_ai_event`
and `announcement_type`. Those require reading the actual 8-K text and are
your RQ2/RQ3 independent variable — per your Synopsis's Data Quality Risk
section, they stay manual and get validated with Cohen's kappa against an
independent re-coder. Automating those would invalidate that check.

**Run this after `03b_cik_ticker_map.ipynb`** (needs the `ticker` column).

## Cell 1 — Clone (or pull) the repo

Run this first, every session. `BASE_DIR` is the single fixed path every
other cell in this notebook reads from and writes to.

In [1]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"      # edit if different
REPO_NAME = "QM640-WALSH-CAPSTONE"        # must match the repo URL EXACTLY (hyphens included)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # set once via Colab Secrets (key icon, left sidebar)

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    # Wipe any stale/incomplete folder from a previous failed attempt before retrying
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

# Fail loudly instead of silently continuing with a non-git folder -
# this is exactly the bug that caused "fatal: not a git repository" earlier.
if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(
        f"Clone failed: no .git folder found at {BASE_DIR}.\n"
        f"Check that GITHUB_USERNAME (\'{GITHUB_USERNAME}\') and REPO_NAME "
        f"(\'{REPO_NAME}\') exactly match your repo URL (case and hyphens "
        f"included), and that GITHUB_TOKEN is set in Colab Secrets with "
        f"Contents: Read and write access."
    )

for sub in ["scripts", "data/raw", "data/processed/returns", "results"]:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

!git -C {BASE_DIR} config user.email "Shan_muganathan@yahoo.com"
!git -C {BASE_DIR} config user.name "Shanmuganathan Ekambaram"

print("Repo ready at:", BASE_DIR)

Cloning into '/content/QM640-WALSH-CAPSTONE'...
remote: Enumerating objects: 910, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 910 (delta 41), reused 62 (delta 23), pack-reused 813 (from 1)
Receiving objects: 100% (910/910), 6.14 MiB | 6.69 MiB/s, done.
Resolving deltas: 100% (482/482), done.
Repo ready at: /content/QM640-WALSH-CAPSTONE


## Cell 2 — Install dependencies

In [2]:
!pip install -q pandas requests yfinance

## Cell 3 — Configuration

In [3]:
import os

RAW_DIR = os.path.join(BASE_DIR, "data/raw")
SCREENING_FILE = os.path.join(RAW_DIR, "screening_worksheet.csv")
HEADERS = {"User-Agent": "QM640 Capstone research Shan_muganathan@yahoo.com"}  # edit to your real email

MIN_TRADING_DAYS = 120
CONFOUND_WINDOW_DAYS = 2   # calendar-day proxy for the +/-2 trading-day window

## Cell 4 — Load the worksheet (must already have `ticker` from Step 3b)

In [4]:
import pandas as pd

df = pd.read_csv(SCREENING_FILE)
df["file_date"] = pd.to_datetime(df["file_date"])

if "ticker" not in df.columns or df["ticker"].isna().all():
    raise RuntimeError(
        "No ticker column found. Run 03b_cik_ticker_map.ipynb first - "
        "the automated checks below need a ticker to pull price data."
    )

print(f"Loaded {len(df)} rows, {df['ticker'].notna().sum()} with a ticker")

Loaded 11676 rows, 10737 with a ticker


## Cell 5 — `confounding_event_flag` (objective: filing-history lookup)

For each unique CIK, pulls that filer's recent submission history from SEC's
own free API and checks whether any *other* filing falls within the
confound window of each event date.

In [5]:
import requests
import time

def get_filing_history(cik):
    """SEC's own free filing-history API, one call per company (cached)."""
    cik_padded = str(int(cik)).zfill(10)
    url = f"https://data.sec.gov/submissions/CIK{cik_padded}.json"
    resp = requests.get(url, headers=HEADERS, timeout=20)
    if resp.status_code != 200:
        return None
    data = resp.json()
    recent = data.get("filings", {}).get("recent", {})
    hist = pd.DataFrame({
        "form": recent.get("form", []),
        "filingDate": pd.to_datetime(recent.get("filingDate", [])),
        "accessionNumber": recent.get("accessionNumber", []),
    })
    return hist


filing_history_cache = {}
confound_flags = []

for _, row in df.iterrows():
    cik = row["cik"]
    if cik not in filing_history_cache:
        filing_history_cache[cik] = get_filing_history(cik)
        time.sleep(0.15)  # stay polite to SEC

    hist = filing_history_cache[cik]
    if hist is None or hist.empty:
        confound_flags.append("REVIEW")  # couldn't fetch - needs manual check
        continue

    window_start = row["file_date"] - pd.Timedelta(days=CONFOUND_WINDOW_DAYS)
    window_end = row["file_date"] + pd.Timedelta(days=CONFOUND_WINDOW_DAYS)
    own_accn = row["accession_no"].split(":")[0] if isinstance(row["accession_no"], str) else None

    nearby = hist[
        (hist["filingDate"] >= window_start) &
        (hist["filingDate"] <= window_end) &
        (hist["accessionNumber"] != own_accn)
    ]
    confound_flags.append("Y" if len(nearby) > 0 else "N")

df["confounding_event_flag"] = confound_flags
print(df["confounding_event_flag"].value_counts())

confounding_event_flag
Y    8694
N    2982
Name: count, dtype: int64


## Cell 6 — `sufficient_history_flag` + trading-halt review flag (objective: price-data lookup)

Caches one wide price pull per ticker (not per event) to keep this fast.

In [6]:
import yfinance as yf
import numpy as np

price_cache = {}
sufficient_flags = []
halt_review_flags = []

tickers = df["ticker"].dropna().unique()
print(f"Pulling price history for {len(tickers)} unique tickers ...")

for i, t in enumerate(tickers):
    try:
        hist = yf.download(t, period="3y", progress=False, auto_adjust=True)
        price_cache[t] = hist
    except Exception as e:
        price_cache[t] = None
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(tickers)}")
    time.sleep(0.1)

for _, row in df.iterrows():
    ticker = row.get("ticker")
    event_date = row["file_date"]

    if pd.isna(ticker) or price_cache.get(ticker) is None or price_cache[ticker].empty:
        sufficient_flags.append("REVIEW")
        halt_review_flags.append("REVIEW")
        continue

    hist = price_cache[ticker]
    pre_event = hist[hist.index < event_date]
    sufficient_flags.append("Y" if len(pre_event) >= MIN_TRADING_DAYS else "N")

    # Crude halt-detection heuristic: look for an unusually large gap between
    # consecutive trading days in the 10 trading days around the event -
    # NOT a definitive halt determination, just a prompt to go look.
    window = hist[(hist.index >= event_date - pd.Timedelta(days=15)) &
                  (hist.index <= event_date + pd.Timedelta(days=15))]
    if len(window) < 2:
        halt_review_flags.append("REVIEW")
    else:
        gaps = window.index.to_series().diff().dt.days.dropna()
        halt_review_flags.append("REVIEW" if (gaps > 5).any() else "N")

df["sufficient_history_flag"] = sufficient_flags
df["trading_halt_flag"] = halt_review_flags
print(df["sufficient_history_flag"].value_counts())
print(df["trading_halt_flag"].value_counts())

Pulling price history for 2036 unique tickers ...


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['NOTEW']: YFPricesMissingError('possibly delisted; no price data found  (period=3y) (Yahoo error = "No data found, symbol may be delisted")')


  50/2036
  100/2036
  150/2036
  200/2036


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['BXCAP']: YFPricesMissingError('possibly delisted; no price data found  (period=3y) (Yahoo error = "No data found, symbol may be delisted")')


  250/2036
  300/2036
  350/2036


...

  1900/2036
  1950/2036


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['FGRS']: YFPricesMissingError('possibly delisted; no price data found  (period=3y) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['OSPR']: YFPricesMissingError('possibly delisted; no price data found  (period=3y) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['ADIG']: YFPricesMissingError('possibly delisted; no price data found  (period=3y) (Yahoo error = "No data found, symbol may be delisted")')


  2000/2036


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['BSTT']: YFPricesMissingError('possibly delisted; no price data found  (period=3y) (Yahoo error = "No data found, symbol may be delisted")')


sufficient_history_flag
Y         7625
N         3024
REVIEW    1027
Name: count, dtype: int64
trading_halt_flag
N         8784
REVIEW    2892
Name: count, dtype: int64


## Cell 7 — Auto-populate `exclude_reason` where an objective flag already justifies it

Leaves it blank where the only open question is `is_genuine_ai_event` /
`announcement_type` — those still need your read of the filing.

In [7]:
def auto_reason(row):
    reasons = []
    if row["confounding_event_flag"] == "Y":
        reasons.append("confounding filing within +/-2 days")
    if row["sufficient_history_flag"] == "N":
        reasons.append("insufficient pre-event price history (<120 trading days)")
    return "; ".join(reasons) if reasons else row.get("exclude_reason", "")


df["exclude_reason"] = df.apply(auto_reason, axis=1)
df.to_csv(SCREENING_FILE, index=False)
print(f"Saved -> {SCREENING_FILE}")
df[["company_name", "confounding_event_flag", "sufficient_history_flag",
    "trading_halt_flag", "exclude_reason"]].head(10)

Saved -> /content/QM640-WALSH-CAPSTONE/data/raw/screening_worksheet.csv


,company_name,confounding_event_flag,sufficient_history_flag,trading_halt_flag,exclude_reason
0,PROGRESS SOFTWARE CORP /MA (PRGS) (CIK 00008...,N,N,REVIEW,insufficient pre-event price history (<120 tra...
1,PEGASYSTEMS INC (PEGA) (CIK 0001013857),N,N,REVIEW,insufficient pre-event price history (<120 tra...
2,OPGEN INC (OPGN) (CIK 0001293818),Y,N,REVIEW,confounding filing within +/-2 days; insuffici...
3,OMNIQ Corp. (OMQS) (CIK 0000278165),N,N,REVIEW,insufficient pre-event price history (<120 tra...
4,"STEM, INC. (STEM) (CIK 0001758766)",N,N,REVIEW,insufficient pre-event price history (<120 tra...
5,"CHART INDUSTRIES INC (GTLS, GTLS-PB) (CIK 00...",Y,N,REVIEW,confounding filing within +/-2 days; insuffici...
6,INSMED Inc (INSM) (CIK 0001104506),Y,N,REVIEW,confounding filing within +/-2 days; insuffici...
7,COMPASS Pathways plc (CMPS) (CIK 0001816590),N,N,REVIEW,insufficient pre-event price history (<120 tra...
8,NEVRO CORP (NVRO) (CIK 0001444380),N,REVIEW,REVIEW,NaN
9,"BurgerFi International, Inc. (BFI, BFIIW) (C...",Y,REVIEW,REVIEW,confounding filing within +/-2 days


## Commit and push results back to GitHub

In [8]:
!git -C {BASE_DIR} add "data/raw/screening_worksheet.csv"
!git -C {BASE_DIR} commit -m "Step 3c: automated objective screening flags (confounding event, sufficient history)"
!git -C {BASE_DIR} push

[main 85dad4f] Step 3c: automated objective screening flags (confounding event, sufficient history)
 1 file changed, 11676 insertions(+), 11676 deletions(-)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (5/5), 152.94 KiB | 1.68 MiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/Shanmuganathan75/QM640-WALSH-CAPSTONE.git
   0f891eb..85dad4f  main -> main


## Summary — what's left for manual review

The columns above are now filled in objectively. **Still required, by hand,
per your Synopsis's methodology:**

- `is_genuine_ai_event` — read the filing, confirm it's a real AI investment/
  partnership/R&D/M&A announcement, not incidental
- `announcement_type` — classify as partnership / R&D / M&A
- Any row where `trading_halt_flag = REVIEW` — a quick look to confirm
  whether the price gap was an actual halt or just a weekend/holiday

This should meaningfully cut down your manual workload — rows already
auto-excluded (`confounding_event_flag = Y` or `sufficient_history_flag =
N`) don't need a filing read at all unless you want to double-check them.

In [9]:
auto_excluded = df["exclude_reason"].str.len() > 0
print(f"Auto-excluded by objective criteria: {auto_excluded.sum()} / {len(df)}")
print(f"Remaining for manual is_genuine_ai_event / announcement_type review: {(~auto_excluded).sum()}")

Auto-excluded by objective criteria: 9497 / 11676
Remaining for manual is_genuine_ai_event / announcement_type review: 2179
